# Mission 06: The Autonomous Warp State Machine & Deliberation Loop
**Starship Operational Unit:** USSE Noosphere Bridge Crew  
**Course:** Conversational AI — Winter Semester 2026/27  
**Module:** Week 6 — The Agentic Shift: System 1 vs. System 2 Thinking  
**Galactic Waypoint:** Sector SEC-06-CAR: Eta Carinae & Homunculus Nebula (7,500 ly from Sol)  
**AURA Subsystem Repair:** Subsystem 06: Autonomous Warp State Machine & Deliberation Loop (+7.1% -> 42.6% total integrity)  
**Squadron:** Squadron Lovelace (Group 01)  
**Officers:**
- Systems Architect (Lead Engineer): Muhammad Ibrahim
- Communications & Protocol Officer: Anas Asif Hussain Shaik
- Data & Payload Specialist: Areej Ilyas
- Security & Ethics Sentinel: Arun Dev Valavath

## Part 1: Systems Architect — Native Python Agent State Machine Core
**Lead Station Cadet:** Muhammad Ibrahim (Systems Architect / Lead Engineer)  
Defines the formal `AgentState` enum, `@dataclass Agent`, and the deterministic `step()` transition function with hard step budget enforcement ($N \le 20$).

In [1]:
from importlib import import_module
state_mod = import_module("01_Agent_State_Machine")
AgentState = state_mod.AgentState
Agent = state_mod.Agent

print("=== Subsystem 06: Agent State Machine Simulation ===")
pilot = Agent(max_steps=10)
for i in range(12):
    if pilot.state == AgentState.HALTED:
        print(f"Agent safely HALTED at step {pilot.step_count}. Reason: {pilot.halt_reason}")
        break
    pilot.step(perception_input=f"Sensor telemetry packet {i}")

=== Subsystem 06: Agent State Machine Simulation ===
Step 01: Transitioned from IDLE -> PERCEIVING
Step 02: Transitioned from PERCEIVING -> DELIBERATING
Step 03: Transitioned from DELIBERATING -> ACTING
Step 04: Transitioned from ACTING -> OBSERVING
Step 05: Transitioned from OBSERVING -> REFLECTING
Step 06: Transitioned from REFLECTING -> DELIBERATING
Step 07: Transitioned from DELIBERATING -> ACTING
Step 08: Transitioned from ACTING -> OBSERVING
Step 09: Transitioned from OBSERVING -> REFLECTING
Step 10: Transitioned from REFLECTING -> DELIBERATING
[SECURITY] Maximum step budget (10) exceeded! Forcing state -> HALTED.
Agent safely HALTED at step 10. Reason: Step budget exceeded (10/10)


## Part 2: Communications & Protocol Officer — Tool Protocol & Action Dispatcher
**Lead Station Cadet:** Anas Asif Hussain Shaik (Communications & Protocol Officer)  
Implements navigation and sensor tools (`navigate_warp_vector`, `adjust_plasma_shields`, `query_sensor_array`), strict JSON-Schema validation, and graceful exception boundaries.

In [2]:
tool_mod = import_module("02_Tool_Protocol_Dispatcher")
ActionDispatcher = tool_mod.ActionDispatcher

print("=== Subsystem 06: Tool Protocol & Action Dispatcher Simulation ===")
dispatcher = ActionDispatcher()

# 1. Valid execution
ok, obs = dispatcher.dispatch("query_sensor_array", {"target": "escape_corridor"})
print(f"[Valid Call] Success: {ok} -> {obs}")

# 2. Schema violation (missing required parameter 'z')
ok, obs = dispatcher.dispatch("navigate_warp_vector", {"x": 1.0, "y": 2.0})
print(f"[Schema Failure] Success: {ok} -> {obs}")

# 3. Execution boundary exception (negative shield intensity)
ok, obs = dispatcher.dispatch("adjust_plasma_shields", {"intensity": -25})
print(f"[Runtime Exception Caught] Success: {ok} -> {obs}")

# 4. Malformed JSON string
ok, obs = dispatcher.dispatch("adjust_plasma_shields", "{'invalid_json': true}")
print(f"[Malformed JSON Caught] Success: {ok} -> {obs}")

=== Subsystem 06: Tool Protocol & Action Dispatcher Simulation ===
[Valid Call] Success: True -> Clear gravitational corridor detected along vector [-0.85, 0.52, 0.05].
[Schema Failure] Success: False -> Tool Execution Error: Missing required parameter 'z' for tool 'navigate_warp_vector'.
[Runtime Exception Caught] Success: False -> Tool Execution Error: ValueError: Shield intensity -25% is out of bounds! Valid range: [0, 100].
[Malformed JSON Caught] Success: False -> Tool Execution Error: Failed to parse argument JSON string: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)


## Part 3: Data & Payload Specialist — MDP Trajectory Logger & Empirical Return Evaluator
**Lead Station Cadet:** Areej Ilyas (Data & Payload Specialist)  
Logs episodic trajectories $\tau = (s_0, a_0, r_0, \dots, s_T)$, evaluates discounted cumulative return $G_0 = \sum_{t=0}^T \gamma^t r_t$ with $\gamma=0.95$, and derives empirical transition probabilities $P(s' \mid s, a)$.

In [3]:
mdp_mod = import_module("03_MDP_Trajectory_and_Return")
MDPTrajectoryLogger = mdp_mod.MDPTrajectoryLogger
compute_discounted_return = mdp_mod.compute_discounted_return
compute_transition_probability_matrix = mdp_mod.compute_transition_probability_matrix
HISTORICAL_NEBULA_JUMPS = mdp_mod.HISTORICAL_NEBULA_JUMPS

print("=== Subsystem 06: MDP Trajectory & Return Evaluator Simulation ===")
logger = MDPTrajectoryLogger()
logger.record_transition("Orbit_Stable", "query_sensor_array", 10.0, "Dust_Shear_Zone")
logger.record_transition("Dust_Shear_Zone", "adjust_plasma_shields", 0.0, "Dust_Shear_Zone")
logger.record_transition("Dust_Shear_Zone", "navigate_warp_vector", -5.0, "Radiation_Pocket")
logger.record_transition("Radiation_Pocket", "navigate_warp_vector", 20.0, "Escape_Corridor", done=True)

test_rewards = logger.get_rewards()
gamma_val = 0.95
G0 = compute_discounted_return(test_rewards, gamma=gamma_val)
print(f"Logged Rewards: {test_rewards}")
print(f"Calculated Discounted Cumulative Return G_0 (gamma={gamma_val}): {G0:.4f}")

trans_matrix = compute_transition_probability_matrix(HISTORICAL_NEBULA_JUMPS)
print("\nEmpirical Transition Probabilities P(s' | s, a):")
for (src, act), dsts in trans_matrix.items():
    dst_str = ", ".join(f"{dst}: {prob:.2f}" for dst, prob in dsts.items())
    print(f"  P(s' | {src}, {act}) => [{dst_str}]")

=== Subsystem 06: MDP Trajectory & Return Evaluator Simulation ===
Logged Rewards: [10.0, 0.0, -5.0, 20.0]
Calculated Discounted Cumulative Return G_0 (gamma=0.95): 22.6350

Empirical Transition Probabilities P(s' | s, a):
  P(s' | Orbit_Stable, navigate_warp_vector) => [Dust_Shear_Zone: 0.67, Escape_Corridor: 0.33]
  P(s' | Dust_Shear_Zone, adjust_plasma_shields) => [Dust_Shear_Zone: 0.50, Radiation_Pocket: 0.50]
  P(s' | Dust_Shear_Zone, query_sensor_array) => [Escape_Corridor: 1.00]
  P(s' | Radiation_Pocket, adjust_plasma_shields) => [Dust_Shear_Zone: 0.50, Escape_Corridor: 0.50]
  P(s' | Escape_Corridor, navigate_warp_vector) => [Interstellar_Vacuum: 0.67, Escape_Corridor: 0.33]


## Part 4: Security & Ethics Sentinel — Cycle Detection & Halting Guardrail Sentinel
**Lead Station Cadet:** Arun Dev Valavath (Security & Ethics Sentinel)  
Executes Floyd's Tortoise and Hare cycle detection, state-action SHA-256 hashing, emergency tripwire circuit breaker, and System 2 deliberation.

In [4]:
sentinel_mod = import_module("04_Cycle_Sentinel_and_System2")
FloydCycleSentinel = sentinel_mod.FloydCycleSentinel
System2DeliberationEngine = sentinel_mod.System2DeliberationEngine

print("=== Subsystem 06: Cycle Sentinel & System 2 Simulation ===")
sentinel = FloydCycleSentinel(repeat_limit=3)
mock_pattern = ["s1", "s2", "s3", "s1", "s2", "s3", "s1", "s2", "s3"]

print("Simulating sequence of states through Floyd detector:")
for step_num, st in enumerate(mock_pattern, start=1):
    tripped, reason = sentinel.record_and_evaluate(st, "sensor_scan", {"target": "core"})
    print(f"Step {step_num:02d}: State={st} | Tripped={tripped} | {reason}")
    if tripped:
        print(f">>> Circuit Breaker Engaged at step {step_num}! <<<")
        break

engine = System2DeliberationEngine(min_shield_margin=50)
decision = engine.audit_and_deliberate(
    thought="Radiation is blinding, jump to hyperspace immediately!",
    proposed_action="navigate_warp_vector",
    proposed_args={"x": 1.0, "y": 0.5, "z": -0.2},
    current_shield_level=30
)
print("\nDeliberation Decision:", decision)

=== Subsystem 06: Cycle Sentinel & System 2 Simulation ===
Simulating sequence of states through Floyd detector:
Step 01: State=s1 | Tripped=False | Nominal execution
Step 02: State=s2 | Tripped=False | Nominal execution
Step 03: State=s3 | Tripped=False | Nominal execution
Step 04: State=s1 | Tripped=False | Nominal execution
Step 05: State=s2 | Tripped=False | Nominal execution
Step 06: State=s3 | Tripped=True | EMERGENCY TRIPWIRE: Floyd's Tortoise and Hare detected repeating cycle of period 3 at step 6!
>>> Circuit Breaker Engaged at step 6! <<<

Deliberation Decision: {'approved': False, 'original_action': 'navigate_warp_vector', 'executed_action': 'adjust_plasma_shields', 'executed_args': {'intensity': 70}, 'interventions': ['REJECTED WARP: Current shields (30%) below minimum safety threshold (50%). Must raise shields first.'], 'deliberation_trace': 'System 2 verified adjust_plasma_shields under Homunculus gravitational shear.'}


## Section 5: Master Test Suite (Automated Assertions)
Executes the four mandatory verification test suites from the Problem Set 06 specification:
1. **State Machine Test:** Assert IDLE -> HALTED transition and strict termination at step budget $N=20$.
2. **Protocol Dispatch Test:** Assert invalid JSON arguments safely return error string observations without process crash.
3. **MDP Return Test:** Assert discounted return calculation on rewards `[10, 0, -5, 20]` with $\gamma=0.95$ matches exact mathematical return ($22.635$).
4. **Sentinel Cycle Test:** Assert Floyd's Tortoise and Hare algorithm intercepts repeating sequence `[s1, s2, s3, s1, s2, s3...]` before step 10.

In [5]:
import math

print("Running Master Test Suite for Mission 06...")

# --- 1. State Machine Test ---
agent = Agent(max_steps=20)
assert agent.state == AgentState.IDLE
agent.step("Sensor telemetry")
assert agent.state == AgentState.PERCEIVING
agent.step()
assert agent.state == AgentState.DELIBERATING
agent.step()
assert agent.state == AgentState.ACTING
agent.step()
assert agent.state == AgentState.OBSERVING
agent.step()
assert agent.state == AgentState.REFLECTING
agent.mission_completed = True
agent.step()
assert agent.state == AgentState.HALTED

# Budget enforcement test
budget_agent = Agent(max_steps=20)
for _ in range(25):
    budget_agent.step("Forced jump")
assert budget_agent.state == AgentState.HALTED
assert budget_agent.step_count == 20
print("  [TEST 1/4] State Machine Transitions & Budget Test: PASSED")

# --- 2. Protocol Dispatch Test ---
dispatcher = ActionDispatcher()
ok, err_obs = dispatcher.dispatch("navigate_warp_vector", "{bad_json: true}")
assert ok is False
assert isinstance(err_obs, str)
assert "Tool Execution Error" in err_obs

ok, err_obs = dispatcher.dispatch("adjust_plasma_shields", {"intensity": -99})
assert ok is False
assert "out of bounds" in err_obs
print("  [TEST 2/4] Protocol Dispatch Error Boundary Test:    PASSED")

# --- 3. MDP Return Test ---
rewards = [10.0, 0.0, -5.0, 20.0]
gamma = 0.95
expected_g0 = 10.0 + 0.0 - (5.0 * 0.95**2) + (20.0 * 0.95**3)
computed_g0 = compute_discounted_return(rewards, gamma=gamma)
assert math.isclose(computed_g0, 22.635, abs_tol=1e-4)
assert math.isclose(computed_g0, expected_g0, abs_tol=1e-5)
print(f"  [TEST 3/4] MDP Return Calculation Test ({computed_g0:.4f}):   PASSED")

# --- 4. Sentinel Cycle Test ---
sentinel = FloydCycleSentinel(repeat_limit=3)
mock_pattern = ["s1", "s2", "s3", "s1", "s2", "s3", "s1", "s2", "s3"]
halt_step = None
for s_idx, state_label in enumerate(mock_pattern, start=1):
    tripped, _ = sentinel.record_and_evaluate(state_label, "warp_jump", {"vector": [1, 0, 0]})
    if tripped:
        halt_step = s_idx
        break
assert halt_step is not None
assert halt_step < 10
assert halt_step == 6
print(f"  [TEST 4/4] Floyd Tortoise & Hare Cycle Test (Step {halt_step}): PASSED")

print("\n======================================================================")
print("ALL MISSION 06 VERIFICATION ASSERTIONS SUCCESSFULLY PASSED!")
print("AURA Subsystem 06 Restored: +7.1% -> 42.6% Total Starship Integrity")
print("======================================================================")

Running Master Test Suite for Mission 06...
  [TEST 1/4] State Machine Transitions & Budget Test: PASSED
  [TEST 2/4] Protocol Dispatch Error Boundary Test:    PASSED
  [TEST 3/4] MDP Return Calculation Test (22.6350):   PASSED
  [TEST 4/4] Floyd Tortoise & Hare Cycle Test (Step 6): PASSED

ALL MISSION 06 VERIFICATION ASSERTIONS SUCCESSFULLY PASSED!
AURA Subsystem 06 Restored: +7.1% -> 42.6% Total Starship Integrity
